# Day-of-year climatology baseline

Forecast = the mean PM₂.₅ on that calendar day of year in the training split. Nothing else.

Train file: `data/modelling/marylebone_train.csv` (2022-04-07 to 2024-12-31).

Scored on the same Random Forest origins as persistence. Run this after `pooled_path_comparison.ipynb`. It writes Climatology rows into `pooled_path_comparison.csv` and rebuilds `pooled_path_comparison_presentable.csv`.

The way this forecast works is using a mean of past observations to predict future values. in this case, the mean PM2.5 for each day of the year (feature) from the training set is used as the forecast for that day into the future.

## Imports and paths

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

HERE = Path(".")
TRAIN_CSV = "../../data/modelling/marylebone_train.csv"
RF_PRED = HERE / "rf_results_path"
CMP_CSV = HERE / "pooled_path_comparison.csv"
PRESENTABLE_CSV = HERE / "pooled_path_comparison_presentable.csv"

## Load training PM₂.₅

In [13]:
train = pd.read_csv(TRAIN_CSV, usecols=["date", "day_of_year", "pm25"])
train["date"] = pd.to_datetime(train["date"])

print(len(train), "train days")
print(train["date"].min().date(), "to", train["date"].max().date())
train.head()

1000 train days
2022-04-07 to 2024-12-31


,date,day_of_year,pm25
0,2022-04-07,97,5.53
1,2022-04-08,98,6.72
2,2022-04-09,99,5.62
3,2022-04-10,100,8.36
4,2022-04-11,101,15.90


## Mean PM₂.₅ by day of year

One number per calendar day (1–366).

In [14]:
clim_by_doy = train.groupby("day_of_year")["pm25"].mean()

print(clim_by_doy.size, "unique days of year")
clim_by_doy.head()

366 unique days of year


day_of_year
1    5.65
2    5.45
3    5.10
4    6.15
5    6.85
Name: pm25, dtype: float64

## Error metrics

Same definitions as `pooled_path_comparison.ipynb`.

In [15]:
def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()
    mse = mean_squared_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-8))) * 100
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mse,
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(y_true, y_pred)),
        "MAPE": float(mape),
        "MBE": float(np.mean(y_pred - y_true)),
    }

## Attach climatology to a prediction table

Look up the training mean for the *target* date's day of year.

In [16]:
def add_climatology(df):
    out = df.copy()
    doy = pd.to_datetime(out["Target_Date"]).dt.dayofyear
    out["Climatology_PM25"] = doy.map(clim_by_doy)
    return out

## Load Random Forest prediction files

These files fix which origins and target dates we score on. We only use actual PM₂.₅ and dates, not the RF predictions.

In [17]:
rf = {}
for h in range(1, 31):
    path = RF_PRED / f"rf_predictions_{h}d.csv"
    df = pd.read_csv(path)
    df["Forecast_Origin"] = pd.to_datetime(df["Forecast_Origin"])
    df["Target_Date"] = pd.to_datetime(df["Target_Date"])
    rf[h] = df

print("leads loaded:", list(rf))
print("1-day rows:", len(rf[1]))
rf[1].head()

leads loaded: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
1-day rows: 363


,Forecast_Origin,Target_Date,Actual_PM25,Predicted_PM25,Persistence_PM25,Residual
0,2025-01-01,2025-01-02,6.4,5.351329,4.0,1.048671
1,2025-01-02,2025-01-03,13.7,8.734338,6.4,4.965662
2,2025-01-03,2025-01-04,13.5,11.004727,13.7,2.495273
3,2025-01-04,2025-01-05,3.0,10.901109,13.5,-7.901109
4,2025-01-05,2025-01-06,9.0,6.737836,3.0,2.262164


## 1-day score

In [18]:
lead1 = add_climatology(rf[1])

row_1d = metrics(lead1["Actual_PM25"], lead1["Climatology_PM25"])
row_1d["Horizon_Days"] = 1
row_1d["Evaluation"] = "lead_1"
row_1d["Model"] = "Climatology"

pd.DataFrame([row_1d]).round(4)


,MAE,MSE,RMSE,R2,MAPE,MBE,Horizon_Days,Evaluation,Model
0,4.6748,53.9383,7.3443,-0.0487,56.9292,-0.6131,1,lead_1,Climatology


## Pooled-path origins

For horizon H, keep only origins that have a complete path of leads 1 through H. Same rule as persistence.

If an origin does not have a complete path of leads 1 through H, it is excluded, so only origins with full lead coverage are scored.

In [19]:
def common_origins(H):
    origin_sets = [set(rf[h]["Forecast_Origin"]) for h in range(1, H + 1)] # create a set of forecast origins for each lead time
    #.intersection will find the common elements across all sets, if an origin is present in every lead time it will be included
    return set.intersection(*origin_sets) # return only the origins that are common across all lead times


def pooled_path_table(H):
    keep = common_origins(H)
    pieces = []
    for h in range(1, H + 1):
        piece = rf[h][rf[h]["Forecast_Origin"].isin(keep)]
        pieces.append(piece[["Forecast_Origin", "Target_Date", "Actual_PM25"]])
    return pd.concat(pieces, ignore_index=True)


print("origins with a full 7-day path:", len(common_origins(7)))
print("origins with a full 14-day path:", len(common_origins(14)))
print("origins with a full 30-day path:", len(common_origins(30)))

origins with a full 7-day path: 357
origins with a full 14-day path: 350
origins with a full 30-day path: 334


## Score 7 / 14 / 30 pooled paths

In [20]:
pooled_rows = []

for H in [7, 14, 30]:
    path = add_climatology(pooled_path_table(H))
    row = metrics(path["Actual_PM25"], path["Climatology_PM25"])
    row["Horizon_Days"] = H
    row["Evaluation"] = "pooled_path"
    row["Model"] = "Climatology"
    pooled_rows.append(row)
    print(H, "path rows:", len(path))

pd.DataFrame(pooled_rows).round(4)


7 path rows: 2499
14 path rows: 4900
30 path rows: 10020


,MAE,MSE,RMSE,R2,MAPE,MBE,Horizon_Days,Evaluation,Model
0,4.6879,54.4603,7.3797,-0.0486,56.9206,-0.6034,7,pooled_path,Climatology
1,4.6517,53.4371,7.3101,-0.0385,57.0210,-0.5108,14,pooled_path,Climatology
2,4.5282,51.2190,7.1567,-0.0314,56.6513,-0.4173,30,pooled_path,Climatology


## Climatology comparison rows

In [21]:
clim_cmp = pd.DataFrame([row_1d] + pooled_rows)
clim_cmp

,MAE,MSE,RMSE,R2,MAPE,MBE,Horizon_Days,Evaluation,Model
0,4.674803,53.938280,7.344269,-0.048650,56.929168,-0.613085,1,lead_1,Climatology
1,4.687851,54.460340,7.379725,-0.048566,56.920632,-0.603365,7,pooled_path,Climatology
2,4.651699,53.437100,7.310068,-0.038462,57.020992,-0.510786,14,pooled_path,Climatology
3,4.528247,51.218956,7.156742,-0.031446,56.651311,-0.417321,30,pooled_path,Climatology


## Write `pooled_path_comparison.csv`

Drop any old Climatology rows, then append these. Safe to re-run.

In [ ]:
if not CMP_CSV.exists():
    raise FileNotFoundError(f"{CMP_CSV} not found. Run pooled_path_comparison.ipynb first.")

comparison = pd.read_csv(CMP_CSV)
comparison = comparison[comparison["Model"] != "Climatology"]
if "Source" in comparison.columns:
    clim_cmp["Source"] = "../../data/modelling/marylebone_train.csv (doy mean; scored on rf_predictions_1d.csv)"
if "Units" in comparison.columns:
    clim_cmp["Units"] = "ug/m3 (MAPE in %)"
comparison = pd.concat(
    [comparison, clim_cmp.reindex(columns=comparison.columns)],
    ignore_index=True,
)
comparison = comparison.sort_values(["Horizon_Days", "MAE"]).reset_index(drop=True)
comparison.to_csv(CMP_CSV, index=False)

print("saved", CMP_CSV.resolve())
print("rows", len(comparison))

saved /Users/matthewbutler/Documents/MastersPaper/code/notebooks/models/pooled_path_comparison.csv
rows 30


## Rebuild the presentable table

This is the same way we did it in `pooled_pwth_comparision` just we need to redo it as otherwise Climatology will not have a `MAE vs Persistence` column or a `RMSE vs Persistence` column.

In [23]:
pers = comparison.loc[
    comparison["Model"] == "Persistence",
    ["Horizon_Days", "MAE", "RMSE"],
].rename(columns={"MAE": "Persistence_MAE", "RMSE": "Persistence_RMSE"})

presentable = comparison.merge(pers, on="Horizon_Days", how="left") # merge the persistence errors into the comparison table so we can calculate skill vs persistence
presentable["MAE_vs_persistence_pct"] = ( # calculate the percentage improvement of MAE over persistence
    (presentable["Persistence_MAE"] - presentable["MAE"])
    / presentable["Persistence_MAE"]
    * 100
)
presentable["RMSE_vs_persistence_pct"] = ( # calculate the percentage improvement of RMSE over persistence
    (presentable["Persistence_RMSE"] - presentable["RMSE"])
    / presentable["Persistence_RMSE"]
    * 100
)

Continue rebuilding the presentable table just this time Climatology will be included so that it also has `MAE vs Persistence` and `RMSE vs Persistence` columns.

In [24]:
rounded = presentable[
    [
        "Model",
        "Horizon_Days",
        "Evaluation",
        "MAE",
        "RMSE",
        "MSE",
        "R2",
        "MAPE",
        "MBE",
        "MAE_vs_persistence_pct",
        "RMSE_vs_persistence_pct",
    ]
].copy()

rounded["MAE"] = rounded["MAE"].round(3)
rounded["RMSE"] = rounded["RMSE"].round(3)
rounded["MSE"] = rounded["MSE"].round(3)
rounded["R2"] = rounded["R2"].round(3)
rounded["MAPE"] = rounded["MAPE"].round(1)
rounded["MBE"] = rounded["MBE"].round(3)
rounded["MAE_vs_persistence_pct"] = rounded["MAE_vs_persistence_pct"].round(1)
rounded["RMSE_vs_persistence_pct"] = rounded["RMSE_vs_persistence_pct"].round(1)

rounded = rounded.rename(
    columns={
        "Horizon_Days": "Horizon (days)",
        "MAPE": "MAPE (%)",
        "MAE_vs_persistence_pct": "MAE vs persistence (%)",
        "RMSE_vs_persistence_pct": "RMSE vs persistence (%)",
    }
)

rounded.to_csv(PRESENTABLE_CSV, index=False)
print("saved", PRESENTABLE_CSV.resolve())
rounded[rounded["Model"] == "Climatology"]

saved /Users/matthewbutler/Documents/MastersPaper/code/notebooks/models/pooled_path_comparison_presentable.csv


,Model,Horizon (days),Evaluation,MAE,RMSE,MSE,R2,MAPE (%),MBE,MAE vs persistence (%),RMSE vs persistence (%)
5,Climatology,1,lead_1,4.675,7.344,53.938,-0.049,56.9,-0.613,-49.6,-29.4
12,Climatology,7,pooled_path,4.688,7.380,54.460,-0.049,56.9,-0.603,4.0,7.8
20,Climatology,14,pooled_path,4.652,7.310,53.437,-0.038,57.0,-0.511,12.8,14.9
28,Climatology,30,pooled_path,4.528,7.157,51.219,-0.031,56.7,-0.417,14.7,15.7
